## Check if there is an overlap in cell IDs between calculated centroids and cellData.csv file

# 1. Load both tables

In [20]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

CELL_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "mibi_tnbc" / "cellData.csv"
CENTROIDS_PATH = PROJECT_ROOT / "data" / "processed" / "mibi_cell_centroids.csv"

cell_df = pd.read_csv(CELL_DATA_PATH)
centroids_df = pd.read_csv(CENTROIDS_PATH)

print("cellData shape:", cell_df.shape)
print("centroids shape:", centroids_df.shape)

cellData shape: (201656, 57)
centroids shape: (249073, 5)


# 2. Create matching keys

In [21]:
cell_keys = (
    cell_df[["SampleID", "cellLabelInImage"]]
    .rename(columns={
        "SampleID": "sample_id",
        "cellLabelInImage": "cell_label"
    })
    .copy()
)

centroid_keys = centroids_df[["sample_id", "cell_label"]].copy()

cell_keys["sample_id"] = cell_keys["sample_id"].astype(int)
cell_keys["cell_label"] = cell_keys["cell_label"].astype(int)

centroid_keys["sample_id"] = centroid_keys["sample_id"].astype(int)
centroid_keys["cell_label"] = centroid_keys["cell_label"].astype(int)

# 3. Check overlap from cellData to masks

#### 
For each cell listed in cellData.csv, did we find a matching object in the segmentation mask?

In [22]:
cell_to_centroid_check = cell_keys.merge(
    centroid_keys,
    on=["sample_id", "cell_label"],
    how="left",
    indicator=True
)

cell_to_centroid_check["_merge"].value_counts()


_merge
both          197678
left_only       3978
right_only         0
Name: count, dtype: int64

In [23]:
print("both = cell exists in cellData and has a matching mask object \n")
print("left_only = cell exists in cellData but no matching mask object was found \n")

both = cell exists in cellData and has a matching mask object 

left_only = cell exists in cellData but no matching mask object was found 



# 4. Important: check only samples 1–41 separately

In [24]:
matched_sample_ids = set(centroids_df["sample_id"].unique())

cell_keys_1_to_41 = cell_keys[cell_keys["sample_id"].isin(matched_sample_ids)].copy()

matched_check = cell_keys_1_to_41.merge(
    centroid_keys,
    on=["sample_id", "cell_label"],
    how="left",
    indicator=True
)

matched_check["_merge"].value_counts()

_merge
both          197678
left_only          0
right_only         0
Name: count, dtype: int64

# 5. Check mask objects that are not in cellData

In [25]:
centroid_to_cell_check = centroid_keys.merge(
    cell_keys,
    on=["sample_id", "cell_label"],
    how="left",
    indicator=True
)

centroid_to_cell_check["_merge"].value_counts()

_merge
both          197678
left_only      51395
right_only         0
Name: count, dtype: int64

In [26]:
print("both       = mask object exists in cellData \n")
print("left_only  = mask object exists but does not appear in cellData: \n background objects \n segmentation artifacts \n filtered-out objects \n objects not included in the final cellData table")


both       = mask object exists in cellData 

left_only  = mask object exists but does not appear in cellData: 
 background objects 
 segmentation artifacts 
 filtered-out objects 
 objects not included in the final cellData table


# 6. Make a clean overlap summary table

In [27]:
summary = pd.DataFrame({
    "check": [
        "cellData rows with matching centroid, all 44 samples",
        "cellData rows without matching centroid, all 44 samples",
        "cellData rows with matching centroid, mask-matched samples only",
        "cellData rows without matching centroid, mask-matched samples only",
        "mask objects with matching cellData row",
        "mask objects without matching cellData row",
    ],
    "n": [
        (cell_to_centroid_check["_merge"] == "both").sum(),
        (cell_to_centroid_check["_merge"] == "left_only").sum(),
        (matched_check["_merge"] == "both").sum(),
        (matched_check["_merge"] == "left_only").sum(),
        (centroid_to_cell_check["_merge"] == "both").sum(),
        (centroid_to_cell_check["_merge"] == "left_only").sum(),
    ]
})

summary

,check,n
0,"cellData rows with matching centroid, all 44 s...",197678
1,"cellData rows without matching centroid, all 4...",3978
2,"cellData rows with matching centroid, mask-mat...",197678
3,"cellData rows without matching centroid, mask-...",0
4,mask objects with matching cellData row,197678
5,mask objects without matching cellData row,51395


In [28]:
# Save summary table
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

summary.to_csv(TABLES_DIR / "cell_mask_overlap_summary.csv", index=False)

# 7. List problematic samples

In [29]:
missing_by_sample = (
    matched_check[matched_check["_merge"] == "left_only"]
    .groupby("sample_id")
    .size()
    .reset_index(name="n_cells_without_centroid")
    .sort_values("n_cells_without_centroid", ascending=False)
)

missing_by_sample

,sample_id,n_cells_without_centroid


In [30]:
# Save list of problematic samples
missing_by_sample.to_csv(
    TABLES_DIR / "cell_rows_without_centroids_by_sample.csv",
    index=False
)

# samples that have no masks

In [31]:
missing_all = cell_to_centroid_check[
    cell_to_centroid_check["_merge"] == "left_only"
]

missing_all["sample_id"].value_counts().sort_index()

sample_id
42    1380
43    1381
44    1217
Name: count, dtype: int64

# 8. Mask-only samples (centroids not in cellData)

Sections 3–7 start from `cellData.csv`, so samples with **zero cell rows** never appear there — even if a segmentation mask exists.

This section breaks down the reverse direction: mask objects that have **no matching row in cellData**. Samples where **all** mask objects are unmatched usually mean the sample is entirely absent from `cellData.csv` (e.g. sample 30).

In [32]:
mask_only_by_sample = (
    centroid_to_cell_check[centroid_to_cell_check["_merge"] == "left_only"]
    .groupby("sample_id")
    .size()
    .reset_index(name="n_mask_objects_without_cellData")
    .sort_values("n_mask_objects_without_cellData", ascending=False)
)

cell_counts = (
    cell_keys.groupby("sample_id")
    .size()
    .reset_index(name="n_cellData_rows")
)

mask_only_summary = mask_only_by_sample.merge(
    cell_counts,
    on="sample_id",
    how="left",
).fillna({"n_cellData_rows": 0})

mask_only_summary["n_cellData_rows"] = mask_only_summary["n_cellData_rows"].astype(int)
mask_only_summary["mask_only_sample"] = mask_only_summary["n_cellData_rows"] == 0

mask_only_summary

,sample_id,n_mask_objects_without_cellData,n_cellData_rows,mask_only_sample
0,30,12128,0,True
1,26,2717,5119,False
2,13,2073,7665,False
3,18,1895,5539,False
4,4,1850,6643,False
5,17,1790,7071,False
6,12,1777,6995,False
7,8,1654,3136,False
8,6,1587,5998,False
9,14,1481,6270,False


In [33]:
fully_mask_only = mask_only_summary[mask_only_summary["mask_only_sample"]].copy()

print("Samples with a mask but zero rows in cellData.csv:")
if fully_mask_only.empty:
    print("  (none)")
else:
    for row in fully_mask_only.itertuples(index=False):
        print(
            f"  Sample {row.sample_id}: "
            f"{row.n_mask_objects_without_cellData:,} mask objects, 0 cellData rows"
        )

fully_mask_only

Samples with a mask but zero rows in cellData.csv:
  Sample 30: 12,128 mask objects, 0 cellData rows


,sample_id,n_mask_objects_without_cellData,n_cellData_rows,mask_only_sample
0,30,12128,0,True


In [34]:
mask_only_summary.to_csv(
    TABLES_DIR / "mask_objects_without_cellData_by_sample.csv",
    index=False,
)

fully_mask_only.to_csv(
    TABLES_DIR / "mask_only_samples_no_cellData.csv",
    index=False,
)

print("Saved:")
print(TABLES_DIR / "mask_objects_without_cellData_by_sample.csv")
print(TABLES_DIR / "mask_only_samples_no_cellData.csv")

Saved:
c:\Users\Kelly\Documents\spatial-ihc-feature-lab\reports\tables\mask_objects_without_cellData_by_sample.csv
c:\Users\Kelly\Documents\spatial-ihc-feature-lab\reports\tables\mask_only_samples_no_cellData.csv
